### Visión por Computadora — TP3

Autor: Nicolás Rodrigues da Cruz (a2123)

In [9]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

IMG_DIR = Path('../Material_TPs/TP3/images')
TEMPLATE_PATH = Path('../Material_TPs/TP3/template/pattern.png')

In [10]:
tpl = cv.imread(str(TEMPLATE_PATH), cv.IMREAD_GRAYSCALE)

In [ ]:
from pathlib import Path
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

IMG_DIR = Path('../Material_TPs/TP3/images')
assert IMG_DIR.exists(), f"No existe carpeta: {IMG_DIR.resolve()}"

names = [
    'coca_logo_1.png',
    'coca_logo_2.png',
    'coca_retro_1.png',
    'coca_retro_2.png',
    'COCA-COLA-LOGO.jpg'
]

names = [n for n in names if (IMG_DIR/n).exists()]
print("Voy a procesar:", names)

tpl = cv.imread(str(TEMPLATE_PATH), cv.IMREAD_GRAYSCALE)
assert tpl is not None, f"No pude leer template: {TEMPLATE_PATH}"

def best_match_single_scale(img_gray, tpl_gray, method=cv.TM_CCOEFF_NORMED):
    R = cv.matchTemplate(img_gray, tpl_gray, method)
    _, maxVal, _, maxLoc = cv.minMaxLoc(R)
    return maxLoc, maxVal

for name in names:
    img_path = IMG_DIR / name
    gray = cv.imread(str(img_path), cv.IMREAD_GRAYSCALE)
    if gray is None:
        print("No pude leer:", img_path.resolve())
        continue

    loc, score = best_match_single_scale(gray, tpl)
    print(f"{name}: score={score:.3f} en loc={loc}")

Voy a procesar: ['coca_logo_1.png', 'coca_logo_2.png', 'coca_retro_1.png', 'coca_retro_2.png', 'COCA-COLA-LOGO.jpg']


error: OpenCV(4.11.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\templmatch.cpp:1175: error: (-215:Assertion failed) _img.size().height <= _templ.size().height && _img.size().width <= _templ.size().width in function 'cv::matchTemplate'


#### Punto 1

In [ ]:
def draw_box(img_rgb, top_left, w, h, color=(0,255,0), text=None):
    x,y = top_left
    cv.rectangle(img_rgb, (x,y), (x+w, y+h), color, 2)
    if text is not None:
        cv.putText(img_rgb, text, (x, max(0,y-8)), cv.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv.LINE_AA)

def best_match_single_scale(img_gray, tpl_gray, method=cv.TM_CCOEFF_NORMED):
    R = cv.matchTemplate(img_gray, tpl_gray, method)
    minVal, maxVal, minLoc, maxLoc = cv.minMaxLoc(R)
    # para TM_CCOEFF_NORMED el mejor es el máximo
    return maxLoc, maxVal, R

def best_match_multiscale(img_gray, tpl_gray, scales=np.linspace(0.6, 1.4, 17), method=cv.TM_CCOEFF_NORMED):
    best = {'score': -1, 'loc': (0,0), 'wh': (tpl_gray.shape[1], tpl_gray.shape[0]), 'scale': 1.0}
    for s in scales:
        tw, th = tpl_gray.shape[1], tpl_gray.shape[0]
        tpl_s = cv.resize(tpl_gray, (int(tw*s), int(th*s)), interpolation=cv.INTER_AREA if s<1 else cv.INTER_CUBIC)
        if tpl_s.shape[0] < 8 or tpl_s.shape[1] < 8:   # evita plantillas demasiado chicas
            continue
        loc, score, _ = best_match_single_scale(img_gray, tpl_s, method)
        if score > best['score']:
            best = {'score': score, 'loc': loc, 'wh': (tpl_s.shape[1], tpl_s.shape[0]), 'scale': s}
    return best

names = [
    'coca_logo_1.png', 'coca_logo_2.png', 'coca_retro_1.png', 'coca_retro_2.png',
    'COCA-COLA-LOGO.jpg', 'pattern.png'
]

THRESH = 0.70  

for name in names:
    img_path = IMG_DIR / name
    bgr = cv.imread(str(img_path))
    if bgr is None:
        print(f'No pude leer {img_path}')
        continue

    gray = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)
    # elegimos multiescala por seguridad (plantilla a diferentes tamaños)
    best = best_match_multiscale(gray, tpl, scales=np.linspace(0.6, 1.6, 21))
    (x,y), (w,h), score = best['loc'], best['wh'], best['score']

    rgb = cv.cvtColor(bgr, cv.COLOR_BGR2RGB)
    if score >= THRESH:
        draw_box(rgb, (x,y), w, h, (0,255,0), text=f'{score:.2f}')
        title = f'{name} — OK (score={score:.2f}, scale={best["scale"]:.2f})'
    else:
        # si el score es bajo, mostramos igual el máximo pero marcado en rojo para no “inventar” detecciones
        draw_box(rgb, (x,y), w, h, (255,0,0), text=f'{score:.2f}')
        title = f'{name} — score bajo (posible FP)'

    plt.figure(figsize=(6,4))
    plt.imshow(rgb); plt.title(title); plt.axis('off'); plt.show()
